In [61]:
import sqlite3
import pandas as pd

conn = sqlite3.connect("kajlia_final.db")

tables = pd.read_sql("SELECT name FROM sqlite_master WHERE type='table'", conn)
print(tables)

                          name
0                        plots
1                        flats
2                 flat_charges
3                        banks
4                   plot_banks
5                   custodians
6              plot_custodians
7                     payments
8          cheque_realizations
9                    audit_log
10        inter_plot_transfers
11                       loans
12           loan_transactions
13                flat_resales
14             refund_payments
15     vendor_flat_adjustments
16             journal_entries
17               journal_lines
18                    settings
19               expense_heads
20                     vendors
21                    expenses
22          internal_transfers
23              flat_discounts
24             journal_effects
25             capital_returns
26                       users
27    cash_deposit_allocations
28             vendor_payments
29  vendor_payment_allocations
30            customer_refunds


In [62]:
flats = pd.read_sql("SELECT * FROM flats", conn)
payments = pd.read_sql("SELECT * FROM payments", conn)

print("FLATS:", flats.shape)
print(flats.dtypes)
print()
print("PAYMENTS:", payments.shape)
print(payments.dtypes)

FLATS: (36, 11)
id                     str
plot_id                str
num                    str
block                  str
customer               str
broker                 str
sale               float64
is_sold              int64
is_plot_owner        int64
owner_party_ids        str
created_at             str
dtype: object

PAYMENTS: (255, 30)
id                        str
plot_id                   str
flat_id                   str
date                      str
amount                float64
mode                      str
description               str
voucher_num               str
cheque_num                str
cheque_bank               str
cheque_date               str
cheque_status             str
bank_id                   str
ref_num                   str
custodian_id              str
created_at                str
from_acc                  str
manual_receipt_num        str
parent_id                 str
is_returned             int64
group_id                  str
cheque_serial         

In [63]:
pd.set_option("display.max_rows", None)
print(payments.dtypes)

id                        str
plot_id                   str
flat_id                   str
date                      str
amount                float64
mode                      str
description               str
voucher_num               str
cheque_num                str
cheque_bank               str
cheque_date               str
cheque_status             str
bank_id                   str
ref_num                   str
custodian_id              str
created_at                str
from_acc                  str
manual_receipt_num        str
parent_id                 str
is_returned             int64
group_id                  str
cheque_serial             str
receipt_book_num          str
cash_exchange_note        str
statement_amount      float64
discount_amount       float64
is_exchange_cheque      int64
handover_type             str
handover_ref_id           str
resale_id                 str
dtype: object


In [64]:
print(payments["cheque_status"].value_counts(dropna=False))
print()
print(payments["mode"].value_counts(dropna=False))

cheque_status
n/a              128
pending           66
realized_bank     52
realized_cash      5
returned           4
Name: count, dtype: int64

mode
Cheque             127
Cash                63
Online Transfer     58
Bank Transfer        7
Name: count, dtype: int64


In [65]:
payments["date"] = pd.to_datetime(payments["date"], errors="coerce")
payments["cheque_date"] = pd.to_datetime(payments["cheque_date"], errors="coerce")

print("date khali:", payments["date"].isnull().sum())
print("cheque_date khali:", payments["cheque_date"].isnull().sum())

cleared = payments[~payments["cheque_status"].isin(["pending", "returned"])]
total_recovery = cleared["amount"].sum()

flat_sale = flats["sale"].sum()
outstanding = flat_sale - total_recovery

print(f"Sale:        {flat_sale:,.0f}")
print(f"Recovery:    {total_recovery:,.0f}")
print(f"Outstanding: {outstanding:,.0f}")

date khali: 0
cheque_date khali: 134
Sale:        1,529,000,000
Recovery:    125,711,100
Outstanding: 1,403,288,900


In [66]:
payments["pending_kind"] = None

pending_mask = payments["cheque_status"] == "pending"

payments.loc[pending_mask & payments["cheque_date"].isnull(), "pending_kind"] = "Khali-date"
payments.loc[pending_mask & (payments["cheque_date"] < pd.Timestamp.today()), "pending_kind"] = "Overdue"
payments.loc[pending_mask & (payments["cheque_date"] >= pd.Timestamp.today()), "pending_kind"] = "Future"

print(payments["pending_kind"].value_counts(dropna=False))
print()
print(payments.groupby("pending_kind")["amount"].sum())

pending_kind
None          189
Future         44
Overdue        15
Khali-date      7
Name: count, dtype: int64

pending_kind
Future        24000000.0
Khali-date     3500000.0
Overdue        8100000.0
Name: amount, dtype: float64


In [67]:
overdue = payments[payments["pending_kind"] == "Overdue"].copy()
overdue["days_overdue"] = (pd.Timestamp.today() - overdue["cheque_date"]).dt.days

overdue["bucket"] = pd.cut(
    overdue["days_overdue"],
    bins=[0, 30, 60, 9999],
    labels=["0-30 days", "31-60 days", "60+ days"]
)

print(overdue.groupby("bucket", observed=True)["amount"].agg(["count", "sum"]))

            count        sum
bucket                      
0-30 days       9  4500000.0
31-60 days      4  2300000.0
60+ days        2  1300000.0


In [68]:
recovery_by_flat = (
    cleared.groupby("flat_id")["amount"]
    .sum()
    .reset_index()
    .rename(columns={"amount": "received"})
)

result = flats.merge(recovery_by_flat, left_on="id", right_on="flat_id", how="left")
result["received"] = result["received"].fillna(0)
result["outstanding"] = result["sale"] - result["received"]

print(result[["num", "sale", "received", "outstanding"]]
      .sort_values("outstanding", ascending=False)
      .head(10))
print(result.shape)

    num        sale   received  outstanding
35  406  50000000.0  1000000.0   49000000.0
25  803  44000000.0        0.0   44000000.0
32  906  44000000.0  1000000.0   43000000.0
33  206  44000000.0  1000000.0   43000000.0
34  201  44000000.0  1611100.0   42388900.0
4   403  44000000.0  2000000.0   42000000.0
27  806  44000000.0  2000000.0   42000000.0
19  606  44000000.0  2000000.0   42000000.0
30  903  44000000.0  3000000.0   41000000.0
22  703  44000000.0  3000000.0   41000000.0
(36, 14)


In [69]:
cleared = cleared.copy()
cleared["month"] = cleared["date"].dt.to_period("M")

monthly = cleared.groupby("month")["amount"].sum()
print(monthly)

month
2026-01    25350000.0
2026-02    22150000.0
2026-03    11200000.0
2026-04    26069600.0
2026-05    18998500.0
2026-06    17900000.0
2026-07     4043000.0
Freq: M, Name: amount, dtype: float64


In [70]:
returned = payments[payments["cheque_status"] == "returned"]
cash = payments[payments["cheque_status"] == "n/a"]

matched = returned.merge(
    cash[["flat_id", "amount"]].drop_duplicates(),
    on=["flat_id", "amount"],
    how="inner"
)

print("Bounced:", returned["amount"].sum())
print("Recovered:", matched["amount"].sum())

Bounced: 1990000.0
Recovered: 1990000.0


In [71]:
pending = payments.loc[payments["cheque_status"] == "pending", "amount"].sum()
bounced = payments.loc[payments["cheque_status"] == "returned", "amount"].sum()
summary = {
    "total_sale": flat_sale,
    "total_recovery": total_recovery,
    "outstanding": outstanding,
    "pending": pending,
    "bounced": bounced,
    "overdue_count": len(overdue),
    "unsecured": outstanding - pending,
}

for k, v in summary.items():
    print(f"{k}: {v:,.0f}")

total_sale: 1,529,000,000
total_recovery: 125,711,100
outstanding: 1,403,288,900
pending: 35,600,000
bounced: 1,990,000
overdue_count: 15
unsecured: 1,367,688,900


In [72]:
import json

print(json.dumps(summary, indent=2))

{
  "total_sale": 1529000000.0,
  "total_recovery": 125711100.0,
  "outstanding": 1403288900.0,
  "pending": 35600000.0,
  "bounced": 1990000.0,
  "overdue_count": 15,
  "unsecured": 1367688900.0
}


In [73]:
text = json.dumps(summary)      # object → text
wapas = json.loads(text)        # text → object

print(type(text))
print(type(wapas))
print(wapas["outstanding"])

<class 'str'>
<class 'dict'>
1403288900.0


In [74]:
data = {
    "project": "Kajlia",
    "total_outstanding": 1403288900,
    "flats": [
        {"num": "A-101", "outstanding": 4500000},
        {"num": "A-102", "outstanding": 3000000}
    ]
}

print(data["project"])
print(data["flats"])
print(data["flats"][0])
print(data["flats"][0]["num"])

Kajlia
[{'num': 'A-101', 'outstanding': 4500000}, {'num': 'A-102', 'outstanding': 3000000}]
{'num': 'A-101', 'outstanding': 4500000}
A-101


In [75]:
for flat in data["flats"]:
    print(flat["num"], flat["outstanding"])

A-101 4500000
A-102 3000000


In [76]:
total = 0
for flat in data["flats"]:
    total = total + flat["outstanding"]

print(total)

7500000


In [77]:
# file mein likho
with open("summary.json", "w") as f:
    json.dump(summary, f, indent=2)

# file se padho
with open("summary.json", "r") as f:
    wapas2 = json.load(f)

print(wapas2["outstanding"])

1403288900.0


In [78]:
import pandas as pd

df = pd.json_normalize(data["flats"])
print(df)

     num  outstanding
0  A-101      4500000
1  A-102      3000000


In [81]:
df2 = pd.json_normalize(
    data,                                      
    record_path="flats",                       
    meta=["project", "total_outstanding"]      
)
print(df2)

     num  outstanding project total_outstanding
0  A-101      4500000  Kajlia        1403288900
1  A-102      3000000  Kajlia        1403288900


In [82]:
df2.sort_values("outstanding", ascending=False)
df2["outstanding"].sum()
df2.groupby("project")["outstanding"].sum()

project
Kajlia    7500000
Name: outstanding, dtype: int64

In [1]:
!pip install python-dotenv

In [2]:
import os
from dotenv import load_dotenv

load_dotenv()
KEY = os.getenv("ANTHROPIC_API_KEY")

print(KEY[:15])

sk-ant-api03-J7
